In [1]:
# Install OpenNMT-py 3.x
!pip3 install OpenNMT-py

  Using cached setuptools-78.1.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached filelock-3.18.0-py3-none-any.whl.metadata (2.9 kB)
  Using cached networkx-3.4.2-py3-none-any.whl.metadata (6.3 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl.m

In [ ]:
# Create the YAML configuration file
# On a regular machine, you can create it manually or with nano
# Note here we are using some smaller values because the dataset is small
# For larger datasets, consider increasing: train_steps, valid_steps, warmup_steps, save_checkpoint_steps, keep_checkpoint

config = '''# config.yaml


## Where the samples will be written
save_data: run

# Training files
data:
    corpus_1:
        path_src: en-zh.en-filtered-wsd-processed.en.subword.train
        path_tgt: en-zh.zh-filtered-wsd.zh.subword.train
        transforms: [filtertoolong]
    valid:
        path_src: en-zh.en-filtered-wsd-processed.en.subword.dev
        path_tgt: en-zh.zh-filtered-wsd.zh.subword.dev
        transforms: [filtertoolong]

# Vocabulary files, generated by onmt_build_vocab
src_vocab: run/source.vocab
tgt_vocab: run/target.vocab

# Vocabulary size - should be the same as in sentence piece
src_vocab_size: 10000
tgt_vocab_size: 10000

# Filter out source/target longer than n if [filtertoolong] enabled
src_seq_length: 512
src_seq_length: 512

# Tokenization options
src_subword_model: source.model
tgt_subword_model: target.model

# Where to save the log file and the output models/checkpoints
log_file: train.log
save_model: models/model.fren

# Stop training if it does not improve after n validations
early_stopping: 4

# Default: 5000 - Save a model checkpoint for each n
save_checkpoint_steps: 2000

# To save space, limit checkpoints to last n
# keep_checkpoint: 3

seed: 3435

# Default: 100000 - Train the model to max n steps 
# Increase to 200000 or more for large datasets
# For fine-tuning, add up the required steps to the original steps
train_steps: 10000

# Default: 10000 - Run validation after n steps
valid_steps: 2000

# Default: 4000 - for large datasets, try up to 8000
warmup_steps: 4000
report_every: 100

# Number of GPUs, and IDs of GPUs
world_size: 1
gpu_ranks: [0]

# Batching
bucket_size: 262144
num_workers: 0  # Default: 2, set to 0 when RAM out of memory
batch_type: "tokens"
batch_size: 4096   # Tokens per batch, change when CUDA out of memory
valid_batch_size: 2048
max_generator_batches: 2
accum_count: [4]
accum_steps: [0]

# Optimization
model_dtype: "fp16"
optim: "adam"
learning_rate: 2
# warmup_steps: 8000
decay_method: "noam"
adam_beta2: 0.998
max_grad_norm: 0
label_smoothing: 0.1
param_init: 0
param_init_glorot: true
normalization: "tokens"
weight_decay: 0.0001

# Model
encoder_type: transformer
decoder_type: transformer
position_encoding: true
enc_layers: 6
dec_layers: 6
heads: 8
hidden_size: 512
word_vec_size: 512
transformer_ff: 2048
dropout_steps: [0]
dropout: [0.1]
attention_dropout: [0.1]
'''

with open("config.yaml", "w+") as config_yaml:
  config_yaml.write(config)

In [ ]:
# Find the number of CPUs/cores on the machine
!nproc --all

In [ ]:
# Build Vocabulary

# -config: path to your config.yaml file
# -n_sample: use -1 to build vocabulary on all the segment in the training dataset
# -num_threads: change it to match the number of CPUs to run it faster

!onmt_build_vocab -config config.yaml -n_sample -1 -num_threads 20

In [ ]:
# Check if the GPU is active
!nvidia-smi -L

In [ ]:
# Check if the GPU is visable to PyTorch
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

gpu_memory = torch.cuda.mem_get_info(0)
print("Free GPU memory:", gpu_memory[0]/1024**2, "out of:", gpu_memory[1]/1024**2)

In [ ]:
# Train the NMT model
!onmt_train -config config.yaml

## Perform Training on SoC Computer Cluster

1. **SSH to your SoC Computer Cluster and copy the train dev test files and config.yaml over**  

2. **Run `salloc` to acquire a GPU host:**  
   ```bash
   salloc -G nv -p gpu-long

3. **Build vocabulary according to number of cores available:**  
   ```bash
   onmt_build_vocab -config config.yaml -n_sample -1 -num_threads 20

4. **Train it on `slurm` by setting time limit to one day:** 
    ```bash
    srun -t 1440 onmt_train -config config.yaml

## Translate

In [ ]:
# Translate the "subworded" source file of the test dataset
# Change the model name, if needed.
!onmt_translate -model models/model.fren_step_10000.pt -src en-zh.en-filtered-wsd-processed.en.subword.test -output zh.translated -gpu 0 -min_length 1

In [5]:
# Check the first 5 lines of the translation file
!head -n 5 zh.translated

▁你 不相信 我 ?
▁我不知道 他们 打算 怎么 处理 这些东西 。
▁它 要求 某些 意义上 的 归 属 感 , ▁因为它 是 一套 没有 指 向 的 系统 。
▁ 自然 的行为 是 守 护 者 。
▁为了 更 自然 地 , ▁我 决定 绘制 科学 地图 , ▁就 在这里 。


In [9]:
# If needed install/update sentencepiece
!pip3 install --upgrade -q sentencepiece

# Desubword the translation file
!python3 ./MT-Preparation/subwording/3-desubword.py ./target.model zh.translated

# Desubword test file
!python3 ./MT-Preparation/subwording/3-desubword.py ./target.model en-zh.zh-filtered-wsd.zh.subword.test


Done desubwording! Output: zh.translated.desubword
Done desubwording! Output: en-zh.zh-filtered-wsd.zh.subword.test.desubword


In [12]:
# Desubword the target file (reference) of the test dataset
# Note: You might as well have split files *before* subwording during dataset preperation, 
# but sometimes datasets have tokeniztion issues, so this way you are sure the file is really untokenized.
!python3 ./MT-Preparation/subwording/3-desubword.py ./source.model en-zh.en-filtered-wsd-processed.en.subword.test

Done desubwording! Output: en-zh.en-filtered-wsd-processed.en.subword.test.desubword


In [11]:
# Check the first 5 lines of the desubworded translation file
!echo "---zh.translated.desubword---"
!head -n 5 zh.translated.desubword

# Check the first 5 lines of the desubworded reference
!echo "---en-zh.zh-filtered-wsd.zh.subword.test.desubword---"
!head -n 5 en-zh.zh-filtered-wsd.zh.subword.test.desubword

---zh.translated.desubword---
你不相信我?
我不知道他们打算怎么处理这些东西。
它要求某些意义上的归属感, 因为它是一套没有指向的系统。
自然的行为是守护者。
为了更自然地, 我决定绘制科学地图, 就在这里。
---en-zh.zh-filtered-wsd.zh.subword.test.desubword---
你不相信我?
我不知道他们将怎么处理那些东西。
这需要有探求精神 因为这整个系统不是以雕塑形式做成的
孩子们天生就会学习
为了让任务更简单一点 我订定了下列科学标准


## Evaluation

In [13]:
# Download the BLEU script
!wget https://raw.githubusercontent.com/ymoslem/MT-Evaluation/main/BLEU/compute-bleu.py

--2025-04-06 15:31:19--  https://raw.githubusercontent.com/ymoslem/MT-Evaluation/main/BLEU/compute-bleu.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 957 [text/plain]
Saving to: ‘compute-bleu.py’

compute-bleu.py     100%[===================>]     957  --.-KB/s    in 0s      

2025-04-06 15:31:19 (13.9 MB/s) - ‘compute-bleu.py’ saved [957/957]



In [14]:
# Install sacrebleu
!pip3 install sacrebleu

In [15]:
# Evaluate the translation (without subwording)
!python3 compute-bleu.py en-zh.zh-filtered-wsd.zh.subword.test.desubword zh.translated.desubword

Reference 1st sentence: 你不相信我?
MTed 1st sentence: 你不相信我?
BLEU:  1.8437310786149796
